# TradingAgents #1518: news and filings after the close

https://github.com/TauricResearch/TradingAgents/issues/1518

A decision dated d kept news until 00:00 UTC of d+1 (20:00 New York in summer) and compared
EDGAR `filed` dates with d, while the decision was scored from the 16:00 close of d. The report's
example: Nike's earnings 8-K accepted 16:15 New York (20:15 UTC) on 2026-10-01, CNBC at
20:07 UTC, Yahoo Finance at 20:19 UTC. The accession number below is a placeholder.

In [1]:
import datetime as dt
import pandas as pd
import fin_pit as pit
from fin_pit.adapters import edgar

news = [
    {"title": "midday preview", "available_at": "2026-10-01T17:00:00Z"},
    {"title": "Nike shares sink in extended trading", "available_at": "2026-10-01T20:07:00Z"},
    {"title": "Nike reports sales below estimates", "available_at": "2026-10-01T20:19:00Z"},
]
submissions = {"filings": {"recent": {
    "accessionNumber": ["0000320187-26-000099"], "filingDate": ["2026-10-01"], "reportDate": ["2026-10-01"],
    "acceptanceDateTime": ["2026-10-01T20:15:00.000Z"], "form": ["8-K"], "items": ["2.02,9.01"],
    "primaryDocument": ["nke-8k.htm"]}}}
filings = edgar.filings(submissions)

## The leak: calendar-day windows

In [2]:
day_start = dt.datetime(2026, 10, 1, tzinfo=dt.timezone.utc)
print([n["title"] for n in news if day_start <= pit.to_utc(n["available_at"]) < day_start + dt.timedelta(days=1)])
print(filings[pd.to_datetime(filings["filing_date"]).dt.date <= dt.date(2026, 10, 1)][["form", "items", "available_at"]])

['midday preview', 'Nike shares sink in extended trading', 'Nike reports sales below estimates']
  form      items              available_at
0  8-K  2.02,9.01 2026-10-01 20:15:00+00:00


## With fin-pit: cut at the decision time, the 16:00 close

In [3]:
with pit.asof("2026-10-01 16:00", tz="America/New_York"):
    print([n["title"] for n in pit.filter(news)])
    print(pit.filter(filings).empty)

['midday preview']
True
